# 🚀 Claude Code Agent no Google Colab (Mapeamento 100% Funcional)

Este notebook contém a adaptação estrutural profunda do Claude Code para o ambiente do Google Colab. 

### 🛠️ O que foi mapeado e adaptado:
1.  **Emulação de TTY**: Forçamos o suporte a cores e interatividade mesmo em terminais limitados.
2.  **Gestão de Memória**: Otimizamos o Node.js para usar até 4GB de RAM, evitando quedas no Colab.
3.  **Privacidade Total**: Desativamos telemetria e tráfego não essencial para maior estabilidade.
4.  **Conexão Direta**: Integração nativa com Ollama (Llama3:8b) sem intermediários.

### 🟢 Célula 1 — Instalação do Ambiente + Servidor Ollama

In [ ]:
%%bash
apt-get update -y
apt-get install -y zstd curl

# Instalar Ollama
curl -fsSL https://ollama.com/install.sh | sh

# Iniciar servidor em segundo plano
nohup /usr/local/bin/ollama serve > ollama.log 2>&1 &
sleep 10

### 🔵 Célula 2 — Preparação do Agente (Auto-Update + Build)

In [ ]:
%%bash
if [ -d "claude-code-agent-colab" ]; then 
  echo "--- Atualizando repositório... ---"
  cd claude-code-agent-colab && git pull origin master
else
  echo "--- Clonando repositório... ---"
  git clone https://github.com/dragonbrxos/claude-code-agent-colab.git
fi

cd claude-code-agent-colab
npm install --no-audit --no-fund --quiet
npm run build

### 🟣 Célula 3 — Configuração Dinâmica de Modelos

In [ ]:
import os
import subprocess

def configurar_modelo(modelo="llama3:8b"):
    print(f"--- Configurando modelo: {modelo} ---")
    subprocess.run(["ollama", "pull", modelo])
    
    # Variáveis de Ambiente Estruturais para o Colab
    os.environ['ANTHROPIC_BASE_URL'] = 'http://localhost:11434/v1'
    os.environ['ANTHROPIC_API_KEY'] = 'ollama'
    os.environ['CLAUDE_MODEL'] = modelo
    
    # Forçar TTY e Cores no Colab
    os.environ['FORCE_COLOR'] = '1'
    os.environ['TERM'] = 'xterm-256color'
    
    # Desativar Telemetria e Tráfego não essencial
    os.environ['DISABLE_TELEMETRY'] = '1'
    os.environ['CLAUDE_CODE_DISABLE_NONESSENTIAL_TRAFFIC'] = '1'
    
    print(f"✅ {modelo} pronto para uso!")

# Execute para o modelo padrão
configurar_modelo("llama3:8b")

### 🚀 Célula 4 — Execução do Agente (Modo Interativo)

In [ ]:
print("Iniciando Claude Code Agent...")
# Usamos flags de memória para o Node.js
!cd claude-code-agent-colab && node --max-old-space-size=4096 dist/main.js